# Hachimi Converter - Colab 训练

在 Colab GPU 上训练 Hachimi 风格转换 Conformer（V3 架构）。

**使用前：**
1. 运行时 → 更改运行时类型 → GPU (A100)
2. 按顺序执行下面的 cell

## 0. 检查 GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("⚠️ 未检测到 GPU，请检查运行时设置")

## 1. 克隆仓库 & 安装依赖

In [ ]:
# TODO: 替换为你的仓库地址
REPO_URL = "https://github.com/7Sageer/hachimi-converter.git"

!git clone {REPO_URL} /content/hachimi-converter
%cd /content/hachimi-converter

In [ ]:
!pip install -q librosa soundfile matplotlib yt-dlp gdown bigvgan

## 2. 准备训练数据

两种方式二选一，**推荐路径 A**（本地处理好再上传，速度快且稳定）。

In [ ]:
### 路径 A：上传本地预处理好的 paired 数据（推荐）
#
# 【本地操作】先在你的 Mac 上执行：
#   cd /path/to/hachimi-converter
#   zip -r paired_data.zip data/paired/
# 然后把 paired_data.zip 上传到 Google Drive（根目录或子目录均可）
#
# 【然后在这里执行】

from google.colab import drive
drive.mount("/content/drive")

import zipfile
from pathlib import Path

zip_path = "/content/drive/MyDrive/hachimi/paired_data.zip"  # 根据实际路径修改
# zip 内部结构为 data/paired/...，所以解压到项目根目录
extract_dir = Path("/content/hachimi-converter")

print("解压中...")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

paired = list((extract_dir / "data" / "paired").glob("*_hach_*.wav"))
print(f"训练片段数: {len(paired)}")

### 路径 B：在 Colab 上从头下载（备用，IP 质量差时容易失败）
# 如果用了路径 A，跳过这三个 cell

%cd /content/hachimi-converter/scripts
!python download_v2.py

In [ ]:
!python scripts/validate_pairs.py

### 路径 B 续

In [ ]:
!python scripts/slice_v2.py

In [ ]:
# 检查训练数据量（路径 B 完成后运行）
import glob
paired = glob.glob("/content/hachimi-converter/data/paired/*_hach_*.wav")
print(f"训练片段数: {len(paired)}")

## 3. 下载 HiFi-GAN 声码器权重

In [ ]:
!python scripts/download_vocoder.py

## 4. 训练 Conformer

In [ ]:
!python scripts/train.py --device cuda --epochs 200 --batch-size 512 --amp --num-workers 2

## 5. 微调 HiFi-GAN 声码器

用哈基米音频数据微调 HiFi-GAN，消除 86Hz 帧率伪影。

In [ ]:
!python scripts/train_vocoder.py --device cuda --epochs 50 --batch-size 32 --amp

## 6. 试听效果

从已下载的原曲中随机选几首，跑推理后直接播放对比。

In [ ]:
import random
from pathlib import Path
from IPython.display import Audio, display, HTML

original_dir = Path("/content/hachimi-converter/data/original")
output_dir = Path("/content/hachimi-converter/output/demo")
output_dir.mkdir(parents=True, exist_ok=True)

# 随机选 3 首原曲（取 15 秒片段）
songs = list(original_dir.glob("*.wav"))
samples = random.sample(songs, min(3, len(songs)))

for song in samples:
    out_path = output_dir / f"{song.stem}_hachimi.wav"
    print(f"\n{'='*60}")
    print(f"转换: {song.stem}")
    !python inference.py "{song}" "{out_path}" 15 30

    display(HTML(f"<h4>{song.stem}</h4>"))
    display(HTML("<b>原曲 (30-45s):</b>"))
    display(Audio(str(song), rate=22050, normalize=False))
    display(HTML("<b>Hachimi 风格:</b>"))
    display(Audio(str(out_path), rate=22050, normalize=False))

## 7. 下载模型权重 & demo 音频

In [ ]:
from google.colab import files
from pathlib import Path

# 下载 U-Net 模型
files.download("/content/hachimi-converter/models/hachimi_unet_best.pt")
files.download("/content/hachimi-converter/models/hachimi_unet_final.pt")

# 下载微调后的 HiFi-GAN vocoder
vocoder_ft = Path("/content/hachimi-converter/models/hifigan/generator_v1_finetuned")
if vocoder_ft.exists():
    files.download(str(vocoder_ft))

# 下载 demo 音频
for f in Path("/content/hachimi-converter/output/demo").glob("*.wav"):
    files.download(str(f))

## 8. (可选) 上传自己的音频测试

In [ ]:
from google.colab import files
uploaded = files.upload()
input_file = list(uploaded.keys())[0]
print(f"已上传: {input_file}")

!python inference.py "/content/{input_file}" /content/output.wav

from IPython.display import Audio
Audio("/content/output.wav")

In [ ]:
files.download("/content/output.wav")